# Ridge, Lasso & Elastic Net Regression

**What this notebook covers:**
- Understanding why plain MLR overfits and how regularization fixes it
- Implementing Ridge from scratch using the closed-form solution β = (XᵀX + αI)⁻¹Xᵀy
- Implementing Lasso from scratch using Coordinate Descent with soft-thresholding
- Comparing Ridge, Lasso, and Elastic Net with scikit-learn
- Visualizing coefficient shrinkage and tuning α with cross-validation

**Prerequisites:**
- Multiple Linear Regression (Normal Equation, multicollinearity, Adjusted R²)
- Basic linear algebra: matrix multiplication, matrix inverse
- Python, NumPy, Pandas, Scikit-learn basics

---

**Dataset:** House Prices — Advanced Regression Techniques  
🔗 [Kaggle Dataset Link](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data)  
*Credits: Kaggle / Dean De Cock — Ames Housing dataset. We reuse the same 8 features as Topic 4's MLR, now adding regularization to see how coefficients shrink and overfitting is controlled.*

In [ ]:
# --- All Imports ---
import numpy as np                        # Matrix operations and regularized OLS math
import pandas as pd                       # Data loading and manipulation
import matplotlib.pyplot as plt           # Plotting
import seaborn as sns                     # Visualizations
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet  # Sklearn models
from sklearn.model_selection import train_test_split, cross_val_score  # Splitting and CV
from sklearn.preprocessing import StandardScaler           # Feature scaling (required!)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score  # Metrics
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)  # Reproducibility for all random operations

## Part 1: Theory Recap

Five key things to remember about Ridge, Lasso, and Elastic Net:

- **Regularization adds a penalty to the OLS loss**: Ridge adds α·Σβⱼ² (L2), Lasso adds α·Σ|βⱼ| (L1), Elastic Net adds a weighted mix of both. The penalty discourages large coefficients, reducing overfitting.
- **Ridge shrinks, Lasso selects**: Ridge shrinks all coefficients smoothly toward zero but never exactly to zero. Lasso can drive some coefficients to exactly zero — automatic feature selection.
- **Ridge has a closed-form solution**: β = (XᵀX + αI)⁻¹Xᵀy. Lasso has no closed form because |βⱼ| is non-differentiable at zero — it needs iterative Coordinate Descent.
- **Standardization is mandatory**: The penalty term is scale-sensitive. Features must be standardized (mean=0, std=1) before fitting, or the penalty will be applied unfairly across features.
- **α controls the bias-variance tradeoff**: α=0 behaves like plain OLS (low bias, high variance). α→∞ shrinks everything toward the mean (high bias, low variance). The right α is found via cross-validation.

## Loading the Dataset

We use the **Ames Housing dataset** — same 8 features as Topic 4's MLR, so we can directly compare:
- **Input features (X):** the same 8 numerical features used in Topic 4
- **Target variable (y):** `SalePrice` — house sale price in USD

Reusing the same features lets us isolate the effect of regularization on the *same* model — Topic 4 (plain MLR) vs Topic 5 (Ridge/Lasso/Elastic Net).

In [ ]:
# Load the Ames Housing dataset
df = pd.read_csv("train.csv")

# Same 8 features as Topic 4 — for direct comparison
features = [
    'Gr Liv Area',      # Above ground living area (sq ft)
    'Overall Qual',     # Overall material and finish quality (1-10)
    'Year Built',       # Original construction year
    'Total Bsmt SF',    # Total basement area (sq ft)
    'Garage Cars',      # Garage size in car capacity
    'Full Bath',        # Number of full bathrooms above ground
    'TotRms AbvGrd',    # Total rooms above ground
    'Fireplaces'        # Number of fireplaces
]
target = 'SalePrice'

df_reg = df[features + [target]].dropna()

print("Dataset Shape:", df_reg.shape)
print("\n--- First 5 Rows ---")
display(df_reg.head())

print("\n--- Statistical Summary ---")
display(df_reg.describe())

## Preprocessing

Regularization makes standardization **non-negotiable**:
1. **Fill missing values** — column medians, same as Topic 4
2. **Train/test split** — 80/20, same random state as Topics 3 & 4 for fair comparison
3. **Standardize all features** — fit scaler on train only, to prevent data leakage
4. **Why scaling matters here specifically** — α penalizes raw coefficient magnitude; an unscaled feature like `Gr Liv Area` (hundreds to thousands) would be penalized completely differently than `Full Bath` (0-4)

In [ ]:
# Step 1: Fill any remaining missing values with column median
for col in features:
    df_reg[col].fillna(df_reg[col].median(), inplace=True)

print("Missing values after fill:", df_reg.isnull().sum().sum())

# Step 2: Separate features and target
X = df_reg[features].values
y = df_reg[target].values

# Step 3: Train/test split — same random_state as Topics 3 & 4 for comparability
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Standardize features — fit only on training data to prevent leakage
# INTERVIEW NOTE: Regularization penalty is scale-dependent — ALWAYS scale before Ridge/Lasso/ElasticNet
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"\nTraining set : {X_train_scaled.shape}")
print(f"Test set     : {X_test_scaled.shape}")
print(f"Features     : {len(features)}")
print(f"\nFeature means after scaling (train): {X_train_scaled.mean(axis=0).round(2)}")
print(f"Feature stds  after scaling (train): {X_train_scaled.std(axis=0).round(2)}")

## Part 2: From Scratch Implementation

We implement **Ridge** using the closed-form solution and **Lasso** using **Coordinate Descent** with soft-thresholding:

```
Ridge:  β = (XᵀX + αI)⁻¹ Xᵀy        (closed form — direct matrix solution)

Lasso:  βⱼ = soft_threshold(ρⱼ, α) / zⱼ   (iterative — updated one coefficient at a time)
        soft_threshold(ρ, α) = sign(ρ) × max(|ρ| - α, 0)
```

For Ridge, note that the identity matrix I has its **first entry (corresponding to β₀) set to 0** — the intercept is never penalized. For Lasso, Coordinate Descent fixes all coefficients except one, finds the optimal value for that one coefficient analytically (which involves the soft-thresholding function), and cycles through all coefficients repeatedly until convergence.

In [ ]:
class RidgeRegressionScratch:
    """
    Ridge Regression from scratch using the closed-form solution.
    β = (XᵀX + αI)⁻¹ Xᵀy
    The intercept (β₀) is NOT penalized.
    """

    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.beta = None
        self.beta_0 = None
        self.beta_coefs = None

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Step 1: Prepend a column of 1s for the intercept
        X_b = np.c_[np.ones((n_samples, 1)), X]

        # Step 2: Build the penalty matrix αI — but DO NOT penalize β₀
        # INTERVIEW NOTE: Setting the (0,0) entry of the penalty matrix to 0
        # ensures the intercept is fit normally, only β₁...βₚ are shrunk
        penalty = self.alpha * np.eye(n_features + 1)
        penalty[0, 0] = 0

        # Step 3: Apply the closed-form Ridge solution
        XtX = X_b.T @ X_b
        Xty = X_b.T @ y
        self.beta = np.linalg.inv(XtX + penalty) @ Xty

        self.beta_0    = self.beta[0]
        self.beta_coefs = self.beta[1:]

    def predict(self, X):
        n_samples = X.shape[0]
        X_b = np.c_[np.ones((n_samples, 1)), X]
        return X_b @ self.beta

    def score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)


class LassoRegressionScratch:
    """
    Lasso Regression from scratch using Coordinate Descent.
    Minimizes: (1/2n) * ||y - Xβ||² + α * Σ|βⱼ|
    The intercept (β₀) is NOT penalized.
    """

    def __init__(self, alpha=1.0, max_iter=1000, tol=1e-4):
        self.alpha = alpha
        self.max_iter = max_iter
        self.tol = tol
        self.beta_0 = None
        self.beta_coefs = None

    @staticmethod
    def _soft_threshold(rho, alpha):
        # soft_threshold(ρ, α) = sign(ρ) * max(|ρ| - α, 0)
        # This is the analytical solution to minimizing a 1D Lasso subproblem
        if rho > alpha:
            return rho - alpha
        elif rho < -alpha:
            return rho + alpha
        else:
            return 0.0

    def fit(self, X, y):
        n_samples, n_features = X.shape

        # Center y so the intercept can be computed separately (not penalized)
        y_mean = np.mean(y)
        y_centered = y - y_mean

        # Initialize coefficients to zero
        beta = np.zeros(n_features)

        # Precompute column norms (X is already standardized, so these are ~n_samples)
        col_norms = np.sum(X ** 2, axis=0)

        # Coordinate Descent — cycle through coefficients until convergence
        for iteration in range(self.max_iter):
            beta_old = beta.copy()

            for j in range(n_features):
                # Residual excluding feature j's current contribution
                residual = y_centered - X @ beta + X[:, j] * beta[j]

                # rho_j = correlation of feature j with the residual
                rho_j = X[:, j] @ residual

                # Update beta_j using soft-thresholding (scaled by alpha * n_samples
                # to match sklearn's convention of (1/2n) loss scaling)
                beta[j] = self._soft_threshold(rho_j, self.alpha * n_samples) / col_norms[j]

            # Check convergence — stop early if coefficients barely changed
            if np.sum(np.abs(beta - beta_old)) < self.tol:
                break

        self.beta_coefs = beta
        self.beta_0 = y_mean  # Intercept = mean of y since X is centered/scaled

    def predict(self, X):
        return self.beta_0 + X @ self.beta_coefs

    def score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)


# Train both from-scratch models with a sample alpha
ridge_scratch = RidgeRegressionScratch(alpha=10.0)
ridge_scratch.fit(X_train_scaled, y_train)

lasso_scratch = LassoRegressionScratch(alpha=1.0)
lasso_scratch.fit(X_train_scaled, y_train)

print("=== Ridge From Scratch (α=10.0) — Coefficients ===")
print(f"β₀ (Intercept) : {ridge_scratch.beta_0:>12,.2f}")
for feat, coef in zip(features, ridge_scratch.beta_coefs):
    print(f"  {feat:20s}: {coef:>12,.2f}")

print("\n=== Lasso From Scratch (α=1.0) — Coefficients ===")
print(f"β₀ (Intercept) : {lasso_scratch.beta_0:>12,.2f}")
for feat, coef in zip(features, lasso_scratch.beta_coefs):
    flag = ' ← zeroed out!' if coef == 0 else ''
    print(f"  {feat:20s}: {coef:>12,.2f}{flag}")

In [ ]:
# Evaluate both from-scratch models on the test set
y_pred_ridge_s = ridge_scratch.predict(X_test_scaled)
y_pred_lasso_s = lasso_scratch.predict(X_test_scaled)

rmse_ridge_s = np.sqrt(mean_squared_error(y_test, y_pred_ridge_s))
r2_ridge_s   = r2_score(y_test, y_pred_ridge_s)

rmse_lasso_s = np.sqrt(mean_squared_error(y_test, y_pred_lasso_s))
r2_lasso_s   = r2_score(y_test, y_pred_lasso_s)

print("=== From Scratch — Test Set Evaluation ===")
print(f"{'Model':12s} {'RMSE':>15s} {'R²':>10s}")
print(f"{'Ridge':12s} ${rmse_ridge_s:>14,.2f} {r2_ridge_s:>10.4f}")
print(f"{'Lasso':12s} ${rmse_lasso_s:>14,.2f} {r2_lasso_s:>10.4f}")

print(f"\nNumber of features zeroed by Lasso: {np.sum(lasso_scratch.beta_coefs == 0)} / {len(features)}")
print("Recall Topic 4's plain MLR used all 8 features with no penalty.")

## Part 3: Sklearn Implementation

Scikit-learn's `Ridge`, `Lasso`, and `ElasticNet` all share the same `.fit(X, y)` / `.predict(X)` API as `LinearRegression`. Internally:
- `Ridge` solves the same closed-form equation we implemented, using a more numerically stable solver
- `Lasso` and `ElasticNet` use Coordinate Descent (like our scratch implementation), but with additional convergence safeguards

We compare all three against plain `LinearRegression` (no penalty) to see the effect of regularization directly.

In [ ]:
# Train plain MLR (baseline, no regularization) plus Ridge, Lasso, Elastic Net
models = {
    'Linear (no penalty)': LinearRegression(),
    'Ridge (α=10)'       : Ridge(alpha=10.0),
    'Lasso (α=1.0)'      : Lasso(alpha=1.0),
    'Elastic Net (α=1.0, l1_ratio=0.5)': ElasticNet(alpha=1.0, l1_ratio=0.5)
}

results = []
coef_table = {'Feature': features}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)
    n_zero = np.sum(np.abs(model.coef_) < 1e-6)

    results.append({'Model': name, 'RMSE': rmse, 'R²': r2, 'Zeroed Coefs': n_zero})
    coef_table[name] = model.coef_.round(2)

print("=== Model Comparison — Test Set ===")
results_df = pd.DataFrame(results)
display(results_df)

print("\n=== Coefficient Comparison Across Models ===")
coef_df = pd.DataFrame(coef_table)
display(coef_df)

# Sanity check: our scratch Ridge should be close to sklearn's Ridge
sklearn_ridge_coefs = models['Ridge (α=10)'].coef_
print("\n=== Scratch vs Sklearn Ridge (α=10) ===")
for feat, c_scratch, c_sklearn in zip(features, ridge_scratch.beta_coefs, sklearn_ridge_coefs):
    match = '✅' if abs(c_scratch - c_sklearn) < 1 else '❌'
    print(f"  {feat:20s}: scratch={c_scratch:>10,.2f} | sklearn={c_sklearn:>10,.2f} {match}")

In [ ]:
# --- Visualisation 1: Coefficient comparison across models ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: Grouped bar chart of coefficients per model
x_pos = np.arange(len(features))
width = 0.2
model_names = list(models.keys())
colors = ['gray', 'steelblue', 'orange', 'green']

for i, name in enumerate(model_names):
    axes[0].bar(x_pos + i * width, coef_table[name], width, label=name, color=colors[i])

axes[0].set_title('Coefficient Comparison: Linear vs Ridge vs Lasso vs Elastic Net',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Feature', fontsize=11)
axes[0].set_ylabel('Coefficient Value (β)', fontsize=11)
axes[0].set_xticks(x_pos + 1.5 * width)
axes[0].set_xticklabels(features, rotation=45, ha='right', fontsize=8)
axes[0].axhline(y=0, color='black', linewidth=1)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3, axis='y')

# Plot 2: Actual vs Predicted for Ridge (best regularized model)
y_pred_ridge = models['Ridge (α=10)'].predict(X_test_scaled)
axes[1].scatter(y_test, y_pred_ridge, alpha=0.4, color='steelblue', s=20)
min_val = min(y_test.min(), y_pred_ridge.min())
max_val = max(y_test.max(), y_pred_ridge.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_title(f'Ridge (α=10) — Actual vs Predicted\nR² = {r2_score(y_test, y_pred_ridge):.4f}',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Actual Sale Price ($)', fontsize=11)
axes[1].set_ylabel('Predicted Sale Price ($)', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Ridge, Lasso & Elastic Net — Ames Housing Dataset', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Visualisation 2: Lasso path — coefficients shrinking to zero as alpha increases ---
alphas_path = np.logspace(-2, 4, 50)
lasso_paths = []

for a in alphas_path:
    lasso_temp = Lasso(alpha=a, max_iter=5000)
    lasso_temp.fit(X_train_scaled, y_train)
    lasso_paths.append(lasso_temp.coef_)

lasso_paths = np.array(lasso_paths)

fig, ax = plt.subplots(figsize=(10, 6))
for i, feat in enumerate(features):
    ax.plot(alphas_path, lasso_paths[:, i], label=feat, linewidth=2)

ax.set_xscale('log')
ax.axhline(y=0, color='black', linewidth=1, linestyle='--')
ax.set_title('Lasso Path — Coefficients Shrinking to Zero as α Increases',
             fontsize=13, fontweight='bold')
ax.set_xlabel('α (log scale)', fontsize=11)
ax.set_ylabel('Coefficient Value (β)', fontsize=11)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

The central hyperparameter for all three models is **α (regularization strength)**. We run two experiments:

1. **α tuning via cross-validation** — sweep α across a log scale and plot validation R² for Ridge and Lasso
2. **Elastic Net l1_ratio sweep** — at a fixed α, vary l1_ratio from 0 (pure Ridge) to 1 (pure Lasso) and observe the transition

These experiments show exactly how α and l1_ratio shape model behavior.

In [ ]:
# Experiment 1: Cross-validated R² vs alpha for Ridge and Lasso
alphas = np.logspace(-2, 3, 30)
ridge_cv_scores = []
lasso_cv_scores = []

for a in alphas:
    ridge_model = Ridge(alpha=a)
    lasso_model = Lasso(alpha=a, max_iter=5000)

    ridge_score = cross_val_score(ridge_model, X_train_scaled, y_train, cv=5, scoring='r2').mean()
    lasso_score = cross_val_score(lasso_model, X_train_scaled, y_train, cv=5, scoring='r2').mean()

    ridge_cv_scores.append(ridge_score)
    lasso_cv_scores.append(lasso_score)

best_ridge_alpha = alphas[np.argmax(ridge_cv_scores)]
best_lasso_alpha = alphas[np.argmax(lasso_cv_scores)]

# Experiment 2: Elastic Net — sweep l1_ratio at a fixed alpha
l1_ratios = np.linspace(0, 1, 11)
elastic_scores = []
elastic_n_zero = []

fixed_alpha = 1.0
for r in l1_ratios:
    # l1_ratio=0 is equivalent to Ridge; ElasticNet requires l1_ratio > 0, so use a tiny value at 0
    r_safe = max(r, 1e-4)
    en_model = ElasticNet(alpha=fixed_alpha, l1_ratio=r_safe, max_iter=5000)
    en_model.fit(X_train_scaled, y_train)
    score = en_model.score(X_test_scaled, y_test)
    elastic_scores.append(score)
    elastic_n_zero.append(np.sum(np.abs(en_model.coef_) < 1e-6))

# Plot both experiments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Alpha tuning curve
axes[0].plot(alphas, ridge_cv_scores, label='Ridge', color='steelblue', linewidth=2, marker='o', markersize=4)
axes[0].plot(alphas, lasso_cv_scores, label='Lasso', color='orange', linewidth=2, marker='s', markersize=4)
axes[0].axvline(x=best_ridge_alpha, color='steelblue', linestyle='--', alpha=0.6,
               label=f'Best Ridge α={best_ridge_alpha:.2f}')
axes[0].axvline(x=best_lasso_alpha, color='orange', linestyle='--', alpha=0.6,
               label=f'Best Lasso α={best_lasso_alpha:.2f}')
axes[0].set_xscale('log')
axes[0].set_title('5-Fold CV R² vs α — Ridge vs Lasso', fontsize=13, fontweight='bold')
axes[0].set_xlabel('α (log scale)', fontsize=11)
axes[0].set_ylabel('Cross-Validated R²', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Elastic Net l1_ratio sweep
axes[1].plot(l1_ratios, elastic_scores, color='green', linewidth=2, marker='o', markersize=6)
axes[1].set_title(f'Elastic Net (α={fixed_alpha}): Test R² vs l1_ratio\n(0=Ridge-like, 1=Lasso-like)',
                  fontsize=13, fontweight='bold')
axes[1].set_xlabel('l1_ratio', fontsize=11)
axes[1].set_ylabel('Test R²', fontsize=11)
axes[1].grid(True, alpha=0.3)
for i, (lr, sc, nz) in enumerate(zip(l1_ratios, elastic_scores, elastic_n_zero)):
    axes[1].annotate(f'{nz} zero', (lr, sc), textcoords='offset points',
                     xytext=(0, 8), ha='center', fontsize=7)

plt.suptitle('Hyperparameter Experiments: α Tuning & l1_ratio Sweep', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Best Ridge α (5-fold CV): {best_ridge_alpha:.4f}  → CV R² = {max(ridge_cv_scores):.4f}")
print(f"Best Lasso α (5-fold CV): {best_lasso_alpha:.4f}  → CV R² = {max(lasso_cv_scores):.4f}")
print(f"\nAs l1_ratio increases from 0 → 1, Elastic Net moves from Ridge-like to Lasso-like behavior,")
print("and the number of zeroed coefficients tends to increase.")

## Part 5: Interview Corner

**Question: Walk me through why Lasso produces sparse models but Ridge does not — geometrically and algebraically.**

This is one of the most common ML theory questions at FAANG interviews. Here is the complete answer:

**Geometric intuition:**  
Both Ridge and Lasso can be viewed as constrained optimization problems:
- Ridge: minimize SSR subject to Σβⱼ² ≤ t (a circular/spherical constraint region)
- Lasso: minimize SSR subject to Σ|βⱼ| ≤ t (a diamond-shaped constraint region with corners on the axes)

The unconstrained OLS solution sits at the center of elliptical error contours. The regularized solution is where these ellipses first touch the constraint region's boundary.
- For Ridge's smooth circular boundary, the touching point is almost never exactly on an axis — so coefficients shrink but stay non-zero
- For Lasso's diamond with sharp corners ON the axes, the elliptical contours are very likely to first touch at a corner — and a corner means one or more βⱼ = 0

**Algebraic intuition:**  
The gradient of the L2 penalty (2αβⱼ) shrinks proportionally to βⱼ — as βⱼ approaches zero, the penalty's pull also approaches zero, so it never quite pushes βⱼ all the way to zero.

The gradient of the L1 penalty is α × sign(βⱼ) — a CONSTANT pull regardless of how small βⱼ is. This constant pull is strong enough to push small coefficients all the way to exactly zero and keep them there (the soft-thresholding operator we implemented from scratch does exactly this).

**Follow-up: When would you choose Elastic Net over either alone?**  
When features are highly correlated (a group of features all measuring "house size", for example), Lasso tends to arbitrarily pick just one feature from the group and zero out the rest — which can be unstable. Elastic Net's L2 component encourages correlated features to share the coefficient "load" more evenly, while its L1 component still allows irrelevant features to be zeroed out.

## Key Takeaways

Remember these 5 bullets for placement interviews:

- **Regularization adds a penalty to the OLS loss to reduce overfitting**: Ridge uses α·Σβⱼ² (L2), Lasso uses α·Σ|βⱼ| (L1), Elastic Net mixes both via l1_ratio. The intercept β₀ is never penalized.
- **Ridge shrinks all coefficients, Lasso can zero them out**: Ridge's closed-form β = (XᵀX + αI)⁻¹Xᵀy always keeps every feature; Lasso's Coordinate Descent with soft-thresholding can set βⱼ = 0 exactly, performing automatic feature selection.
- **Standardization before fitting is mandatory**: Because the penalty term depends on raw coefficient magnitude, unscaled features would be penalized unevenly based on units rather than importance.
- **α controls the bias-variance tradeoff**: α=0 reduces to plain OLS (low bias, high variance, risk of overfitting); α→∞ shrinks all βⱼ toward zero (high bias, low variance, risk of underfitting). The optimal α is found via cross-validation, not guessed.
- **Elastic Net is the safe default for correlated, high-dimensional data**: It combines Ridge's stability with correlated features and Lasso's ability to perform feature selection — tuned via both α and l1_ratio.